# 5.5 训练容错与弹性训练 (Fault Tolerance & Elastic Training)

> 🕐 预估学习时间：35分钟

大规模 LLM 训练通常持续数周甚至数月，涉及成百上千张 GPU。在如此规模下，硬件故障、网络异常、软件 Bug 几乎不可避免。训练容错与弹性训练技术确保训练在故障发生后能够快速恢复，并支持动态调整集群规模，从而最大化资源利用率与训练稳定性。

本节涵盖：
- 故障类型与故障模拟
- Checkpoint 管理与恢复
- 自动故障检测与恢复
- 弹性训练（动态扩缩容）
- 冗余计算与高可用（RPO/RTO）

## 1. 训练容错概述

**为什么需要容错？**

在大规模训练中，故障不是例外而是常态：
- **GPU 故障**：数千张 GPU 中，单卡 MTBF（平均故障间隔）可能仅数天
- **网络故障**：NCCL 超时、交换机抖动、网络分区
- **显存溢出（OOM）**：激活内存波动、内存碎片化
- **软件 Bug**：算子崩溃、数值异常、死锁

**故障影响**：
- 一次未保存的故障可能损失数小时甚至数天的训练进度
- 大模型训练成本极高（单次训练可达数百万美元），故障代价巨大
- 手动恢复耗时且易出错

**核心目标**：将故障导致的进度损失（RPO）与恢复时间（RTO）控制在可接受范围内。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import random
from collections import deque, Counter

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# 故障模拟器：模拟 GPU 故障、网络分区、OOM 等事件
class FailureSimulator:
    def __init__(self, failure_rate=0.05, seed=42):
        self.failure_rate = failure_rate
        self.rng = random.Random(seed)
        self.failure_log = deque(maxlen=100)
        self.failure_types = ['gpu_failure', 'network_partition', 'oom', 'software_bug']

    def inject_failure(self, step):
        roll = self.rng.random()
        if roll < self.failure_rate:
            ftype = self.rng.choice(self.failure_types)
            self.failure_log.append((step, ftype, time.time()))
            return ftype
        return None

    def stats(self):
        counts = Counter(ft for _, ft, _ in self.failure_log)
        return dict(counts)

# 训练状态：跟踪进度与检查点信息
class TrainingState:
    def __init__(self, total_steps=1000):
        self.total_steps = total_steps
        self.current_step = 0
        self.loss_history = []
        self.checkpoint_steps = []
        self.failed = False
        self.failure_count = 0

    def step(self, loss):
        self.current_step += 1
        self.loss_history.append(loss)

    def checkpoint(self):
        self.checkpoint_steps.append(self.current_step)

    def progress(self):
        return self.current_step / self.total_steps

simulator = FailureSimulator(failure_rate=0.08, seed=42)
state = TrainingState(total_steps=200)

for step_idx in range(1, state.total_steps + 1):
    loss = 2.0 * np.exp(-step_idx / 80.0) + 0.1 * np.random.randn()
    state.step(float(loss))
    if step_idx % 50 == 0:
        state.checkpoint()
    failure = simulator.inject_failure(step_idx)
    if failure:
        state.failed = True
        state.failure_count += 1
        break

print('=== Failure Simulation ===')
print(f'Total steps planned: {state.total_steps}')
print(f'Steps completed: {state.current_step}')
print(f'Progress: {state.progress():.1%}')
print(f'Checkpoints saved: {len(state.checkpoint_steps)} at steps {state.checkpoint_steps}')
print(f'Failure occurred: {state.failed}')
print(f'Failure count: {state.failure_count}')
print(f'\nFailure type distribution: {simulator.stats()}')
print(f'Last loss: {state.loss_history[-1]:.4f}')
print(f'\nKey: At scale, failures are inevitable; training systems must detect,')
print(f'contain, and recover from them without losing significant progress.')

## 2. Checkpoint 管理与恢复

**检查点策略**：
- **全量检查点（Full Checkpoint）**：保存模型参数、优化器状态、调度器、随机数状态等完整信息；恢复简单但 I/O 开销大
- **增量检查点（Incremental Checkpoint）**：仅保存相对上次的变化；节省存储与 I/O，但恢复需重建完整状态
- **频率权衡**：保存越频繁，故障损失越小，但 I/O 开销与存储成本越高

**恢复流程**：
1. 定位最近的有效检查点
2. 校验检查点完整性（防止加载损坏文件）
3. 恢复模型、优化器、数据迭代器状态
4. 从检查点步数继续训练

**分布式一致性**：多节点检查点需保证所有 rank 保存同一 step 的状态，避免部分节点成功导致的状态不一致。

In [ ]:
import torch
import torch.nn as nn
import os
import time
import tempfile

torch.manual_seed(42)

# 检查点管理器：支持全量/增量检查点与版本控制
class CheckpointManager:
    def __init__(self, model, optimizer, save_dir=None):
        self.model = model
        self.optimizer = optimizer
        self.save_dir = save_dir or os.path.join(tempfile.gettempdir(), 'ckpt_demo')
        self.versions = []
        os.makedirs(self.save_dir, exist_ok=True)

    def save_full(self, step, extra=None):
        ckpt = {
            'step': step,
            'model_state': self.model.state_dict(),
            'optimizer_state': self.optimizer.state_dict(),
            'timestamp': time.time(),
            'extra': extra or {},
        }
        path = os.path.join(self.save_dir, f'full_step{step}.pt')
        torch.save(ckpt, path)
        self.versions.append({'step': step, 'type': 'full', 'path': path})
        return path

    def save_incremental(self, step, base_step, delta=None):
        ckpt = {
            'step': step,
            'base_step': base_step,
            'delta': delta if delta is not None else self.model.state_dict(),
            'timestamp': time.time(),
        }
        path = os.path.join(self.save_dir, f'incr_step{step}.pt')
        torch.save(ckpt, path)
        self.versions.append({'step': step, 'type': 'incremental', 'path': path})
        return path

    def load(self, path):
        ckpt = torch.load(path, weights_only=False)
        if 'model_state' in ckpt:
            self.model.load_state_dict(ckpt['model_state'])
            self.optimizer.load_state_dict(ckpt['optimizer_state'])
        return ckpt

# 检查点恢复：检测最近有效检查点并校验一致性
class CheckpointRecovery:
    def __init__(self, manager):
        self.manager = manager

    def find_last_good(self):
        for ver in reversed(self.manager.versions):
            try:
                ckpt = torch.load(ver['path'], weights_only=False)
                if self._validate(ckpt):
                    return ver
            except Exception:
                continue
        return None

    def _validate(self, ckpt):
        if 'model_state' in ckpt:
            return all(torch.is_tensor(v) for v in ckpt['model_state'].values())
        return 'delta' in ckpt

    def resume(self):
        ver = self.find_last_good()
        if ver is None:
            print('No valid checkpoint found; starting from scratch.')
            return 0
        ckpt = self.manager.load(ver['path'])
        ver_type = ver['type']
        ckpt_step = ckpt['step']
        print(f'Resumed from {ver_type} checkpoint at step {ckpt_step}')
        return ckpt_step

model = nn.Linear(16, 4)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
cm = CheckpointManager(model, optimizer)
recovery = CheckpointRecovery(cm)

# 模拟训练并定期保存检查点
for step in [10, 20, 30]:
    x = torch.randn(8, 16)
    y = model(x).sum()
    y.backward()
    optimizer.step()
    optimizer.zero_grad()
    if step % 20 == 0:
        cm.save_full(step, extra={'loss': float(y)})
    else:
        cm.save_incremental(step, base_step=0)

print('=== Checkpoint Management ===')
print(f'Versions saved: {len(cm.versions)}')
for v in cm.versions:
    vstep = v['step']
    vtype = v['type']
    vpath = os.path.basename(v['path'])
    print(f'  step {vstep}: {vtype} -> {vpath}')

resumed_step = recovery.resume()
print(f'\nResumed at step: {resumed_step}')
print(f'\nKey: Frequent checkpoints reduce replay cost but add I/O overhead;')
print(f'incremental checkpoints and validation ensure fast, safe recovery.')

## 3. 自动故障检测与恢复

**心跳监控**：
- 每个 worker 定期向监控器发送心跳
- 超时未收到心跳判定为故障

**健康检查**：
- Loss 异常飙升（loss spike）可能指示数值不稳定
- 梯度范数爆炸（gradient explosion）需及时干预
- GPU 利用率、显存、温度等硬件指标

**自动恢复**：
- 检测到故障后自动回滚到最近良好检查点
- 重启训练进程，跳过损坏数据
- 设置最大重启次数，避免无限重启循环

**Watchdog 进程**：独立于训练进程的守护进程，负责监控与触发恢复，避免训练进程自身崩溃导致监控失效。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
from collections import deque

torch.manual_seed(42)

# 健康监控：心跳与异常检测（loss spike、梯度爆炸）
class HealthMonitor:
    def __init__(self, window=10, loss_spike_threshold=2.0, grad_norm_threshold=100.0):
        self.window = window
        self.loss_history = deque(maxlen=window)
        self.heartbeats = {}
        self.loss_spike_threshold = loss_spike_threshold
        self.grad_norm_threshold = grad_norm_threshold

    def heartbeat(self, worker_id):
        self.heartbeats[worker_id] = time.time()

    def check_heartbeat(self, timeout=5.0):
        now = time.time()
        dead = [wid for wid, t in self.heartbeats.items() if now - t > timeout]
        return dead

    def record_loss(self, loss):
        self.loss_history.append(loss)

    def detect_anomaly(self, grad_norm=None):
        if len(self.loss_history) >= self.window:
            recent = sorted(self.loss_history)
            median = recent[len(recent) // 2]
            if self.loss_history[-1] > median * self.loss_spike_threshold:
                return 'loss_spike'
        if grad_norm is not None and grad_norm > self.grad_norm_threshold:
            return 'grad_explosion'
        return None

# 自动恢复：故障检测、检查点校验、训练重启
class AutoRecovery:
    def __init__(self, monitor, max_restarts=3):
        self.monitor = monitor
        self.max_restarts = max_restarts
        self.restart_count = 0
        self.last_good_step = 0

    def handle_failure(self, anomaly, step):
        if self.restart_count >= self.max_restarts:
            return 'give_up'
        self.restart_count += 1
        print(f'  [AutoRecovery] anomaly={anomaly} at step {step}; restart #{self.restart_count}')
        return 'restarted'

    def mark_good(self, step):
        self.last_good_step = step

monitor = HealthMonitor(window=10, loss_spike_threshold=2.0, grad_norm_threshold=100.0)
recovery = AutoRecovery(monitor, max_restarts=3)

# 模拟 4 个 worker 心跳
for wid in range(4):
    monitor.heartbeat(wid)
# 模拟 worker 0 心跳超时
monitor.heartbeats[0] = time.time() - 10.0

print('=== Auto Fault Detection & Recovery ===')
print(f'Workers reporting: {len(monitor.heartbeats)}')
print(f'Dead workers (timeout=5s): {monitor.check_heartbeat(timeout=5.0)}')

# 模拟训练过程，注入 loss spike 与梯度爆炸
np.random.seed(42)
losses = list(np.linspace(2.0, 0.5, 12)) + [5.0, 0.45, 0.4]
final_step = 0
for i, loss in enumerate(losses, start=1):
    monitor.record_loss(float(loss))
    grad_norm = 10.0 if i != 13 else 200.0
    anomaly = monitor.detect_anomaly(grad_norm=grad_norm)
    if anomaly:
        action = recovery.handle_failure(anomaly, i)
        if action == 'restarted':
            continue
    recovery.mark_good(i)
    final_step = i

print(f'\nTotal restarts: {recovery.restart_count}')
print(f'Last good step: {recovery.last_good_step}')
print(f'Final step reached: {final_step}')
print(f'\nKey: Continuous health monitoring plus automated rollback lets')
print(f'training self-heal from transient faults without human intervention.')

## 4. 弹性训练 (Elastic Training)

**动机**：传统分布式训练要求 world_size 固定，节点故障或加入需重启整个作业。弹性训练允许训练过程中动态调整 GPU 数量。

**PyTorch Elastic / TorchRun**：
- `--nnodes=MIN:MAX` 指定节点数范围
- Rendezvous 机制实现节点发现与协调
- 节点加入/离开时自动重新分片数据与重建进程组

**关键挑战**：
- **数据重分片**：world_size 变化后需重新划分数据，避免重复或遗漏
- **Batch size 调整**：全局 batch size 可能变化，影响训练动力学
- **学习率调整**：部分实现按 world_size 缩放学习率
- **状态同步**：所有 rank 需在重分片点同步

**适用场景**：抢占式实例（preemptible instances）、弹性云集群、故障频发环境。

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

# 弹性训练器：动态管理 world size，缩放时重新分配数据
class ElasticTrainer:
    def __init__(self, model, total_data_size=2000):
        self.model = model
        self.total_data_size = total_data_size
        self.world_size = 1
        self.step = 0
        self.samples_done = 0
        self.local_data_size = 0
        self.history = []

    def rescale(self, new_world_size):
        old_ws = self.world_size
        self.world_size = new_world_size
        per_worker = math.ceil(self.total_data_size / new_world_size)
        self.local_data_size = per_worker
        self.history.append((self.step, old_ws, new_world_size))
        print(f'  [Elastic] rescale: {old_ws} -> {new_world_size} GPUs; per-worker={per_worker}')

    def train_step(self, batch_size=32):
        self.step += 1
        throughput = batch_size * self.world_size
        self.samples_done += throughput
        return throughput

    def progress(self):
        return min(1.0, self.samples_done / self.total_data_size)

# 弹性会合：节点发现与协调
class ElasticRendezvous:
    def __init__(self, min_nodes=1, max_nodes=8):
        self.min_nodes = min_nodes
        self.max_nodes = max_nodes
        self.active_nodes = set()
        self.round = 0

    def join(self, node_id):
        self.active_nodes.add(node_id)
        return self.check_status()

    def leave(self, node_id):
        self.active_nodes.discard(node_id)
        return self.check_status()

    def check_status(self):
        n = len(self.active_nodes)
        if n >= self.min_nodes:
            self.round += 1
            return ('ready', n, self.round)
        return ('waiting', n, self.round)

model = nn.Linear(32, 8)
trainer = ElasticTrainer(model, total_data_size=2000)
rdv = ElasticRendezvous(min_nodes=1, max_nodes=8)

print('=== Elastic Training ===')
# 初始 1 GPU
trainer.rescale(1)
for _ in range(3):
    trainer.train_step()

# 扩容到 4 GPU
for nid in range(4):
    status, n, r = rdv.join(nid)
print(f'Rendezvous after scale-up: {status}, nodes={n}, round={r}')
trainer.rescale(4)
for _ in range(3):
    trainer.train_step()

# 缩容到 2 GPU（模拟节点离开）
rdv.leave(2)
rdv.leave(3)
status, n, r = rdv.check_status()
print(f'Rendezvous after scale-down: {status}, nodes={n}, round={r}')
trainer.rescale(2)
for _ in range(3):
    trainer.train_step()

print(f'\nTotal steps: {trainer.step}')
print(f'Samples processed: {trainer.samples_done}')
print(f'Progress: {trainer.progress():.1%}')
print(f'Rescale events: {len(trainer.history)}')
for s, old, new in trainer.history:
    print(f'  step {s}: {old} -> {new} GPUs')
print(f'\nKey: Elastic training lets world size change mid-run, improving')
print(f'cluster utilization and surviving node churn without restart.')

## 5. 冗余计算与高可用

**冗余计算**：通过额外资源复制关键计算，在主副本故障时由备份接管。

**主备复制（Primary-Backup）**：
- 主节点执行训练，备份节点定期同步状态
- 主节点故障时，备份提升为新主节点
- 同步频率决定数据损失量

**关键指标**：
- **RPO（Recovery Point Objective）**：恢复点目标，即允许丢失的最大数据量（用步数或时间衡量）
- **RTO（Recovery Time Objective）**：恢复时间目标，即从故障到恢复服务的最大允许时间

**权衡**：
- 同步越频繁 → RPO 越小，但同步开销越大
- 冗余越多 → 可用性越高，但资源成本越高
- 通常对关键训练任务采用适度冗余，而非全量复制

In [ ]:
import torch
import torch.nn as nn
import time
import copy

torch.manual_seed(42)

# 高可用训练器：主备复制、状态同步、故障转移
class HighAvailabilityTrainer:
    def __init__(self, model, sync_interval=10):
        self.primary = model
        self.backup = copy.deepcopy(model)
        self.sync_interval = sync_interval
        self.last_sync_step = 0
        self.failover_count = 0
        self.rto_seconds = 0.0

    def train_step(self, x, step):
        out = self.primary(x).sum()
        out.backward()
        if step - self.last_sync_step >= self.sync_interval:
            self._sync()
            self.last_sync_step = step
        return float(out)

    def _sync(self):
        self.backup.load_state_dict(self.primary.state_dict())

    def failover(self, step):
        t0 = time.time()
        self.primary = copy.deepcopy(self.backup)
        rpo_steps = step - self.last_sync_step
        self.rto_seconds = time.time() - t0 + 0.05
        self.failover_count += 1
        return rpo_steps

    def compute_rpo(self, step):
        return step - self.last_sync_step

model = nn.Linear(64, 16)
ha = HighAvailabilityTrainer(model, sync_interval=5)

print('=== High Availability & Redundancy ===')
# 模拟训练
for step in range(1, 21):
    x = torch.randn(4, 64)
    ha.train_step(x, step)
    rpo = ha.compute_rpo(step)
    if step % 5 == 0:
        print(f'  step {step}: synced, current RPO={rpo} steps')

# 在 step 23 触发故障转移（上次同步在 step 20）
step = 23
rpo_steps = ha.failover(step)
print(f'\nFailover at step {step}:')
print(f'  RPO (data loss): {rpo_steps} steps (max acceptable: {ha.sync_interval})')
print(f'  RTO (recovery time): {ha.rto_seconds*1000:.1f} ms')
print(f'  Failover count: {ha.failover_count}')

# 演示更频繁同步降低 RPO
ha2 = HighAvailabilityTrainer(nn.Linear(64, 16), sync_interval=2)
for s in range(1, 11):
    ha2.train_step(torch.randn(4, 64), s)
rpo2 = ha2.failover(11)
print(f'\nWith sync_interval=2: RPO={rpo2} steps (vs {rpo_steps} with interval=5)')

print(f'\nKey: Primary-backup replication bounds RPO (data loss) and RTO')
print(f'(downtime); tighter sync lowers RPO at higher overhead cost.')

## 📝 课后思考题

1. 在大规模训练中，全量检查点与增量检查点各有什么优劣？如何根据场景选择？
2. 自动故障检测中，如何区分瞬时抖动与真正故障？误报与漏报各自有什么代价？
3. 弹性训练在 world_size 变化时为何需要重新分片数据？如果处理不当会出现什么问题？
4. RPO 与 RTO 分别从哪个维度衡量容错能力？在实际项目中如何平衡二者与成本？

---
> 本节涵盖了训练容错与弹性训练的核心概念与代码实现。建议结合实际集群环境，设计合适的检查点策略与恢复机制，并通过故障注入实验验证系统的鲁棒性。